# 04 - Idiot Sandwich: TRL DPOTrainer + LoRA on SmolLM2-360M-Instruct

In [ ]:
# ---- Task 1: repository + environment setup (Kaggle, GPU T4 x2) ----
# Private repo: add a GitHub token as a Kaggle secret named GITHUB_TOKEN
# (Add-ons -> Secrets -> Add secret, then tick it for this notebook).
REPO_URL = "https://github.com/zainabusman22/advanced-genai-pa2.git"
REPO = "/kaggle/working/advanced-genai-pa2"

import os, subprocess, sys
if not os.path.exists(REPO):
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    subprocess.run(["git", "clone", "--depth", "1", authed, REPO], check=True, capture_output=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)  # don't keep the token on disk
    del token, authed
os.chdir(REPO)
!git -C {REPO} log --oneline -1
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
!nvidia-smi

In [ ]:
!python -m pip install -q "trl>=0.19,<0.24" "peft>=0.13"

In [ ]:
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Select the GPU T4 x2 accelerator"

In [ ]:
!python scripts/train_lora_dpo.py --output_dir outputs/lora_dpo_persona

## Generate persona-eval outputs: instruct model (baseline) vs LoRA-DPO

In [ ]:
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --template tokenizer \
    --eval_set eval_sets/persona_style_eval_24.json --generator_name smollm2-360m-instruct \
    --output_path outputs/generations/persona/instruct.json --max_new_tokens 256
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --template tokenizer \
    --adapter_path outputs/lora_dpo_persona/adapter --eval_set eval_sets/persona_style_eval_24.json \
    --generator_name smollm2-360m-lora-dpo-persona --output_path outputs/generations/persona/lora_dpo.json --max_new_tokens 256
!python scripts/build_pairs.py --model_path outputs/generations/persona/lora_dpo.json \
    --baseline_path outputs/generations/persona/instruct.json --eval_kind persona \
    --output_path outputs/eval/persona_lora_vs_instruct_pairs.jsonl
!python scripts/plot_metrics.py --lora outputs/lora_dpo_persona/metrics.jsonl --out_dir report/figures

In [ ]:
import json
for a, b in zip(json.load(open("outputs/generations/persona/instruct.json"))[:6],
                json.load(open("outputs/generations/persona/lora_dpo.json"))[:6]):
    print("PROMPT:", a["instruction"]); print("  INSTRUCT:", a["output"][:300]); print("  LORA-DPO:", b["output"][:300]); print()

In [ ]:
# Archive outputs so they survive the session (then: Save Version -> attach as dataset).
!cd {REPO} && zip -qr /kaggle/working/lora_persona_outputs.zip outputs/lora_dpo_persona outputs/generations/persona outputs/eval report/figures
!ls -lh /kaggle/working/*.zip